# Back to School (1986) – Thornton Melon’s Economics Lesson as a Linear Programming Problem

This notebook summarizes a fun exercise: turning the iconic classroom debate from *Back to School* (Rodney Dangerfield as Thornton Melon vs. the stuffy economics professor) into a **Linear Programming (LP)** model.

In the scene, the professor builds a clean theoretical factory for "tape recorders" (later switched to "widgets"), calculating only legitimate costs. Thornton interrupts with gritty real-world extras:
- Greasing politicians
- Kickbacks to carpenters
- Chats with the teamsters (unions)
- Payoffs to building inspectors
- Mob handling waste disposal ("I assure you it's not the Boy Scouts")

He also notes Japanese manufacturers would "kill us on the labor costs."

We model minimizing total factory cost → then show infeasibility under Japanese competition → now we extend to **logistics**: shipping widgets to warehouses at minimum transportation cost.

In [ ]:
import pulp

# ────────────────────────────────────────────────
# Original Thornton-style factory cost model
# ────────────────────────────────────────────────

prob = pulp.LpProblem("Tape_Recorder_Factory", pulp.LpMinimize)
x = pulp.LpVariable("x", lowBound=0)          # factory size / capacity
p = pulp.LpVariable("p", lowBound=0)          # units produced

prob += 119 * x + 0.02 * p + 10               # objective: total cost
prob += p >= 1000                             # meet minimum order
prob += p <= 1000 * x                         # production ≤ capacity

prob.solve(pulp.PULP_CBC_CMD(msg=0))

print("Status:", pulp.LpStatus[prob.status])
print(f"Total Cost ($1,000s): {pulp.value(prob.objective):.2f}")
print(f"Factory size: {pulp.value(x):.1f}")
print(f"Produced: {pulp.value(p):.0f}")

## Adding Japanese Competition Constraint

Thornton: "The Japs will kill us on the labor costs!"

Constraint: average cost per unit ≤ $15 (Japanese benchmark).  
→ 119x + 0.02p + 10 ≤ 0.015p  
→ 119x + 10 ≤ -0.005p

This makes the problem **infeasible** for p ≥ 1000 – capturing the dialogue perfectly!

In [ ]:
# With Japanese constraint
prob_jap = pulp.LpProblem("With_Japanese_Competition", pulp.LpMinimize)
x_j = pulp.LpVariable("x_j", lowBound=0)
p_j = pulp.LpVariable("p_j", lowBound=0)

prob_jap += 119 * x_j + 0.02 * p_j + 10
prob_jap += p_j >= 1000
prob_jap += p_j <= 1000 * x_j
prob_jap += 119 * x_j + 10 <= -0.005 * p_j   # Japanese avg cost cap

status = prob_jap.solve(pulp.PULP_CBC_CMD(msg=0))
print("Status:", pulp.LpStatus[status])
if status != 1:
    print("INFEASIBLE – The Japs will kill us on the labor costs!")

## Logistics Extension: Transportation Problem – Shipping Widgets to Warehouses

Thornton’s factory (or factories) is now producing widgets efficiently — but someone has to pay to ship them to regional warehouses.

Classic **Transportation Problem** (one of the oldest LP applications):
- Minimize total shipping cost
- Meet warehouse demand without exceeding factory supply

Setup (inspired by textbook examples but Thornton-flavored):
- 2 factories: New York (high labor, close to East Coast) and Memphis (cheaper, central)
- 4 warehouses: New York, Chicago, Atlanta, Los Angeles
- Supply: NY = 450 units, Memphis = 700 units
- Demand: NY=300, Chicago=200, Atlanta=250, LA=400
- Shipping costs ($/unit) vary by distance + Thornton's "real-world" surcharges (tolls, union deals, etc.)

In [ ]:
# ────────────────────────────────────────────────
# Transportation Problem – Widget Shipping
# ────────────────────────────────────────────────

factories = ['New_York', 'Memphis']
warehouses = ['NY', 'Chicago', 'Atlanta', 'LA']

# Supply (units available)
supply = {'New_York': 450, 'Memphis': 700}

# Demand (units required)
demand = {'NY': 300, 'Chicago': 200, 'Atlanta': 250, 'LA': 400}

# Cost matrix [$/unit] – Thornton style: includes tolls, kickbacks, fuel...
costs = {
    ('New_York', 'NY'):      4,
    ('New_York', 'Chicago'): 18,
    ('New_York', 'Atlanta'): 22,
    ('New_York', 'LA'):      45,
    ('Memphis',  'NY'):      25,
    ('Memphis',  'Chicago'): 12,
    ('Memphis',  'Atlanta'): 14,
    ('Memphis',  'LA'):      28
}

# Create problem
trans = pulp.LpProblem("Widget_Transportation", pulp.LpMinimize)

# Variables: units shipped from factory i to warehouse j
ship = pulp.LpVariable.dicts("ship", (factories, warehouses), lowBound=0, cat='Continuous')

# Objective: minimize total transport cost
trans += pulp.lpSum(costs[(i,j)] * ship[i][j] for i in factories for j in warehouses)

# Constraints
# 1. Supply limits
for i in factories:
    trans += pulp.lpSum(ship[i][j] for j in warehouses) <= supply[i], f"Supply_{i}"

# 2. Demand satisfaction
for j in warehouses:
    trans += pulp.lpSum(ship[i][j] for i in factories) == demand[j], f"Demand_{j}"

# Solve
trans.solve(pulp.PULP_CBC_CMD(msg=0))

print("Status:", pulp.LpStatus[trans.status])
print(f"Total Shipping Cost: ${pulp.value(trans.objective):,.2f}")
print("\nOptimal Shipments:")
for i in factories:
    for j in warehouses:
        val = pulp.value(ship[i][j])
        if val > 0.01:
            print(f"  {i:10} → {j:8} : {val:6.0f} units   @ ${costs[(i,j)]}/unit")

print(f"\nTotal shipped from New York: {sum(pulp.value(ship['New_York'][j]) for j in warehouses):.0f}")
print(f"Total shipped from Memphis:   {sum(pulp.value(ship['Memphis'][j]) for j in warehouses):.0f}")

## Interpretation – Thornton’s Take

- Memphis dominates long-haul shipments (cheaper central location + lower "union handling" fees).
- New York factory mostly serves local NY demand (avoids high cross-country trucking costs).
- Total cost is minimized — but Thornton would probably say: "Still too much goin' to tolls and teamsters! Maybe buy a few politicians and build our own rail line…"

## What-if scenarios

1. Fuel price spike → increase all costs by 30% → re-solve → expect more reliance on closer shipments.
2. LA demand surges to 600 → add constraint → may become infeasible unless supply increases.
3. Add a third factory in Texas → Thornton would approve: "More locations, more leverage!"

Feel free to modify supplies, demands, or costs and re-run!

## Conclusion

The math proves Thornton right: legitimate costs + real-world bribes + high U.S. labor make tape recorders uncompetitive against Japan in 1986. Switch to widgets... or lease space and invest the down payment (as Thornton suggests)!

Now we see the full picture: production + distribution. No respect for pure theory — but plenty for good old-fashioned optimization! 😂